# Trinity THLP 50-Item MC Test Run
DeepMind AGI Hackathon 2026

In [ ]:
# Install dependencies
!pip install protobuf==5.29.6 --quiet
!pip install -q kaggle-benchmarks kaggle

# Import all required modules
import os
os.environ["RENDER_SUBRUNS"] = "False"

import kaggle_benchmarks as kbench
import kaggle
import pandas as pd
import glob
from dataclasses import dataclass

print("✅ All imports successful")

In [ ]:
# MC Answer schema and matching function
@dataclass
class MCAnswer:
    answer: str

def match_mc_answer(response: str, expected: str) -> bool:
    """MC format: compare first letter only (A, B, C, D)."""
    if not response or not expected:
        return False
    got = response.strip().upper()[0]
    exp = str(expected).strip().upper()[0]
    return got == exp

print("✅ MC matching function defined")

In [ ]:
# Download and load MC dataset
print("📥 Downloading MC dataset...")
!mkdir -p /kaggle/working/datasets

kaggle.api.dataset_download_files(
    'playra/trinity-cognitive-probes-thlp-mc',
    path='/kaggle/working/datasets/',
    unzip=True
)

csv_files = glob.glob('/kaggle/working/datasets/**/*.csv', recursive=True)
csv_path = [f for f in csv_files if 'thlp_mc.csv' in f][0]

print(f"📂 Using: {csv_path}")

df = pd.read_csv(csv_path)
mc_df = df[df['question_type'] == 'mc'].copy()
eval_df = mc_df.head(50)  # 50-item test run

print(f"📊 Loaded {len(eval_df)} MC questions (from {len(df)} total rows)")

In [ ]:
# Define inner MC task
@kbench.task(name="trinity_thlp_50item", store_task=False)
def thlp_50item(llm, question, choices, expected_answer) -> bool:
    prompt = f"""{question}\n\n{choices}\n\nAnswer with ONLY option letter (A, B, C, or D)."""
    response = llm.prompt(prompt, schema=MCAnswer)
    return match_mc_answer(response.answer, expected_answer)

print("✅ Inner task registered")

In [ ]:
# Define outer benchmark task
@kbench.task(
    name="Trinity THLP 50-Item Test Run",
    description="MC benchmark for hippocampal learning with 50-item test run."
)
def thlp_test_50items(llm) -> float:
    with kbench.client.enable_cache():
        runs = thlp_50item.evaluate(
            llm=[llm], evaluation_data=eval_df, n_jobs=2,
            timeout=180, max_attempts=1, remove_run_files=True,
        )
    results_df = runs.as_dataframe()
    valid = results_df[results_df["result"].notna()]
    if len(valid) == 0:
        kbench.assertions.assert_true(False, expectation="No valid results")
        return 0.0
    accuracy = float(valid["result"].mean())
    kbench.assertions.assert_true(
        True, expectation=f"THLP MC accuracy: {accuracy:.2%} ({len(valid)}/{len(eval_df)})"
    )
    return accuracy

print("✅ Outer benchmark task registered")

In [ ]:
# Run the benchmark
run = thlp_test_50items.run(llm=kbench.llm)
print(f"\\n🏆 Result: {run.result:.2%} (50 items)")

%choose thlp_test_50items